## Load the Libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Initialize Data Folder
data_folder_path = "../data/"

# Load the Training Data

In [3]:
train_dataset = pd.read_excel(data_folder_path+"train_dataset.xlsx")

In [4]:
train_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4928 entries, 0 to 4927
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        4928 non-null   object 
 1   gender            4928 non-null   object 
 2   SeniorCitizen     4928 non-null   int64  
 3   Partner           4928 non-null   object 
 4   Dependents        4928 non-null   object 
 5   tenure            4928 non-null   int64  
 6   PhoneService      4928 non-null   object 
 7   MultipleLines     4928 non-null   object 
 8   InternetService   4928 non-null   object 
 9   OnlineSecurity    4928 non-null   object 
 10  OnlineBackup      4928 non-null   object 
 11  DeviceProtection  4928 non-null   object 
 12  TechSupport       4928 non-null   object 
 13  StreamingTV       4928 non-null   object 
 14  StreamingMovies   4928 non-null   object 
 15  Contract          4928 non-null   object 
 16  PaperlessBilling  4928 non-null   object 


In [5]:
train_dataset.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,1336-EZFZY,Female,0,No,No,4,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.05,83.30,No
1,2162-FRZAA,Male,0,Yes,Yes,63,No,No phone service,DSL,No,...,Yes,Yes,No,No,Two year,No,Bank transfer (automatic),39.35,2395.05,No
2,0504-HHAPI,Female,1,No,No,27,Yes,Yes,Fiber optic,No,...,Yes,No,No,Yes,Month-to-month,No,Credit card (automatic),88.30,2467.75,Yes
3,4909-JOUPP,Male,1,Yes,No,72,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),109.70,7898.45,No
4,3082-VQXNH,Male,0,Yes,No,3,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,No,Credit card (automatic),29.80,94.40,No


## Fill NA values with 0 for Total Charges as we have already checked that these are new customers with tenure as 0

In [6]:
# Fill na with 0
train_dataset = train_dataset.fillna(0)

In [7]:
train_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4928 entries, 0 to 4927
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        4928 non-null   object 
 1   gender            4928 non-null   object 
 2   SeniorCitizen     4928 non-null   int64  
 3   Partner           4928 non-null   object 
 4   Dependents        4928 non-null   object 
 5   tenure            4928 non-null   int64  
 6   PhoneService      4928 non-null   object 
 7   MultipleLines     4928 non-null   object 
 8   InternetService   4928 non-null   object 
 9   OnlineSecurity    4928 non-null   object 
 10  OnlineBackup      4928 non-null   object 
 11  DeviceProtection  4928 non-null   object 
 12  TechSupport       4928 non-null   object 
 13  StreamingTV       4928 non-null   object 
 14  StreamingMovies   4928 non-null   object 
 15  Contract          4928 non-null   object 
 16  PaperlessBilling  4928 non-null   object 


## Clean the feature names and make it consistent

In [8]:
# PaymentMethod have some inconsistent value names
train_dataset["PaymentMethod"].value_counts()

PaymentMethod
Electronic check             1651
Mailed check                 1163
Bank transfer (automatic)    1083
Credit card (automatic)      1031
Name: count, dtype: int64

In [9]:
train_dataset["PaymentMethod"] = train_dataset["PaymentMethod"].replace("Bank transfer (automatic)", "Bank transfer").replace("Credit card (automatic)", "Credit card")
train_dataset["PaymentMethod"].value_counts()

PaymentMethod
Electronic check    1651
Mailed check        1163
Bank transfer       1083
Credit card         1031
Name: count, dtype: int64

In [10]:
train_dataset.columns = [col.upper() for col in train_dataset.columns.to_list()]

In [11]:
# Remove Churn and CustomerID
# Split dataset into features and target variable
X_train = train_dataset.drop(labels = ["CHURN","CUSTOMERID"],axis=1)
Y_train = train_dataset["CHURN"]

print(f"Shape of data ",X_train.shape)
print(f"Shape of target variable ",Y_train.shape)

Shape of data  (4928, 19)
Shape of target variable  (4928,)


## Feature engineering

In [12]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [13]:
## Extract numerical columns
numerical_cols = X_train.select_dtypes(include = ["number"]).columns
print(f"Numerical Columns: {numerical_cols}")
print(f"Total Numerical Columns : ",len(numerical_cols))

## Extract categorical columns
categorical_cols = X_train.select_dtypes(include = ["object"]).columns
print(f"Categorical Columns: {categorical_cols}")
print(f"Total Categorical Columns : ",len(categorical_cols))

Numerical Columns: Index(['SENIORCITIZEN', 'TENURE', 'MONTHLYCHARGES', 'TOTALCHARGES'], dtype='object')
Total Numerical Columns :  4
Categorical Columns: Index(['GENDER', 'PARTNER', 'DEPENDENTS', 'PHONESERVICE', 'MULTIPLELINES',
       'INTERNETSERVICE', 'ONLINESECURITY', 'ONLINEBACKUP', 'DEVICEPROTECTION',
       'TECHSUPPORT', 'STREAMINGTV', 'STREAMINGMOVIES', 'CONTRACT',
       'PAPERLESSBILLING', 'PAYMENTMETHOD'],
      dtype='object')
Total Categorical Columns :  15


In [14]:
# Create a transformer for categocial columns
preprocessor = ColumnTransformer(transformers=[("CAT",OneHotEncoder(drop='first'), categorical_cols)],remainder='passthrough')

In [15]:
preprocessor.fit(X=X_train)

# Get the output feature names (e.g., ['', 'city_LA'])
output_feature_names = [f"{col.replace("remainder__","").replace(" ", "_").upper()}" for col in preprocessor.get_feature_names_out()]
print(output_feature_names)
print(len(output_feature_names))

['CAT__GENDER_MALE', 'CAT__PARTNER_YES', 'CAT__DEPENDENTS_YES', 'CAT__PHONESERVICE_YES', 'CAT__MULTIPLELINES_NO_PHONE_SERVICE', 'CAT__MULTIPLELINES_YES', 'CAT__INTERNETSERVICE_FIBER_OPTIC', 'CAT__INTERNETSERVICE_NO', 'CAT__ONLINESECURITY_NO_INTERNET_SERVICE', 'CAT__ONLINESECURITY_YES', 'CAT__ONLINEBACKUP_NO_INTERNET_SERVICE', 'CAT__ONLINEBACKUP_YES', 'CAT__DEVICEPROTECTION_NO_INTERNET_SERVICE', 'CAT__DEVICEPROTECTION_YES', 'CAT__TECHSUPPORT_NO_INTERNET_SERVICE', 'CAT__TECHSUPPORT_YES', 'CAT__STREAMINGTV_NO_INTERNET_SERVICE', 'CAT__STREAMINGTV_YES', 'CAT__STREAMINGMOVIES_NO_INTERNET_SERVICE', 'CAT__STREAMINGMOVIES_YES', 'CAT__CONTRACT_ONE_YEAR', 'CAT__CONTRACT_TWO_YEAR', 'CAT__PAPERLESSBILLING_YES', 'CAT__PAYMENTMETHOD_CREDIT_CARD', 'CAT__PAYMENTMETHOD_ELECTRONIC_CHECK', 'CAT__PAYMENTMETHOD_MAILED_CHECK', 'SENIORCITIZEN', 'TENURE', 'MONTHLYCHARGES', 'TOTALCHARGES']
30


In [16]:
# Transform and update the data with column names
transformed_data = preprocessor.transform(X_train)
print(transformed_data.shape)
X_train_transformed = pd.DataFrame(
            data=transformed_data,
            columns=output_feature_names,
            index=X_train.index,
        )




(4928, 30)


In [17]:
X_train_transformed.describe()

,CAT__GENDER_MALE,CAT__PARTNER_YES,CAT__DEPENDENTS_YES,CAT__PHONESERVICE_YES,CAT__MULTIPLELINES_NO_PHONE_SERVICE,CAT__MULTIPLELINES_YES,CAT__INTERNETSERVICE_FIBER_OPTIC,CAT__INTERNETSERVICE_NO,CAT__ONLINESECURITY_NO_INTERNET_SERVICE,CAT__ONLINESECURITY_YES,...,CAT__CONTRACT_ONE_YEAR,CAT__CONTRACT_TWO_YEAR,CAT__PAPERLESSBILLING_YES,CAT__PAYMENTMETHOD_CREDIT_CARD,CAT__PAYMENTMETHOD_ELECTRONIC_CHECK,CAT__PAYMENTMETHOD_MAILED_CHECK,SENIORCITIZEN,TENURE,MONTHLYCHARGES,TOTALCHARGES
count,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,...,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000
mean,0.509334,0.484375,0.300325,0.900365,0.099635,0.418222,0.433239,0.224026,0.224026,0.282265,...,0.210836,0.237825,0.593141,0.209213,0.335024,0.235998,0.159903,32.179586,64.163535,2248.304698
std,0.499964,0.499807,0.458446,0.299543,0.299543,0.493317,0.495573,0.416982,0.416982,0.450147,...,0.407944,0.425794,0.491298,0.406788,0.472047,0.424664,0.366553,24.525388,30.214025,2257.695022
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,18.250000,0.000000
25%,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,33.600000,372.312500
50%,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,29.000000,70.050000,1372.750000
75%,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,55.000000,89.650000,3707.112500
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,72.000000,118.750000,8684.800000


## Creating a load and transformer pipeline to do all the steps

In [18]:
import os
import logging
def make_logger(name: str, log_file: str = None) -> logging.Logger:
    logger = logging.getLogger(f"{__name__}.{name}")

    if not logger.handlers:
        level = logging.INFO
    
        # Add File Handler if configured (for local debugging/MLflow artifacts)
        if log_file:
            file_handler = logging.FileHandler(log_file, mode='a')
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            file_handler.setFormatter(formatter)
            logger.addHandler(file_handler)
        logger.setLevel(level)
    
    return logger



In [99]:
class LoadDataset:
    """Class for loading the data"""
    def __init__(self,data_folder:str,log_file: str = "../logs/data_loader.log"):
        """Initializes the data folder
        """
        self.data_folder = data_folder
        self.logger = make_logger("DataLoader",log_file)
    
    def load_excel(self,file_name:str) -> pd.DataFrame:
        """Loads the excel file. Currently support .xlsx files with single sheet only.
        """
        self.logger.info(f"Loading Dataset")
        file_path = self.data_folder + file_name
        try:
            df = pd.read_excel(file_path, engine='openpyxl')
            self.logger.info(f"[LOG] Successfully loaded {len(df)} rows from {file_path}")
            return df
        except Exception as e:
            self.logger.info(f"[ERROR] Failed to read the Excel file: {e}")
            raise IOError(f"Failed to read the Excel file: {e}")
    
    def clean_data(self, dataset:pd.DataFrame, target_col: str)->pd.DataFrame:
        """Clean the dataset and rename any fields if required.
        """
        self.logger.info(f"Cleaning Dataset")
        try:
            # Separate features and target variable
            X = dataset.drop(columns=[target_col]) if target_col else dataset.copy()
            y = dataset[target_col].values if target_col else None

            # Rename Column Values
            X["PaymentMethod"] = X["PaymentMethod"].replace({
                "Bank transfer (automatic)": "Bank transfer",
                "Credit card (automatic)": "Credit card"
            })
            X=X.drop("customerID",axis=1)
            # Rename Columns to UPPERCASE
            X.columns = [col.upper() for col in X.columns.to_list()]
            if(y is not None):
                self.logger.info(f"Cleaning Dataset Complete. Shape of features {X.shape} & Shape of target Variable {len(y)}")
            else:
                self.logger.info(f"Cleaning Dataset Complete. Shape of features {X.shape}")
            
            return X, y
        except Exception as e:
            raise ValueError(f"Failed to clean the dataset {e}")
        

In [100]:
from typing import Any

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from pathlib import Path
import pickle
from dataclasses import dataclass

@dataclass
class PreprocessorState:
    """
    Container to hold the fitted model and necessary metadata for inference.
    """
    transformer: Any = None
    target_col: str = None
    logger: Any = None

    def save(self,file_path:str):
        """Saves the state to a pickle file."""
        self.logger.info(f"Saving Preprocessor")
        if(file_path):
            try:
                path = Path(file_path)
                path.parent.mkdir(parents=True, exist_ok=True)
                
                with open(file_path, 'wb') as f:
                    pickle.dump(self, f)
                
                self.logger.info("[LOG] Preprocessor saved successfully at file_path: {file_path}")
                return True
            except Exception as e:
                self.logger.exception(f"[ERROR] Failed to save the preprocessor at file_path: {file_path} with Error: {e}")
                raise
        else:
            self.logger.exception("[ERROR] Filepath is incorrect: {file_path}")

    def load(self, file_path: str):
        """Loads the state from a pickle file."""
        self.logger.info(f"Loading Preprocessor")
        try:
            with open(file_path, 'rb') as f:
                self.__init__(transformer=None) # Reset first
                loaded_state = pickle.load(f)
            
            # Unpack attributes safely
            self.transformer = loaded_state.transformer
            self.target_col = loaded_state.target_col
            
            self.logger.info(f"[LOG] Model loaded successfully from: {file_path}")
            return True
        except Exception as e:
            self.logger.exception(f"[ERROR] Failed to load model: {e}")
            raise


class FeatureExtraction:
    """Class for extracting the features from the data. This includes scaling, null value imputer and categorical encoding"""
    def __init__(self, preprocessor_state_obj = None, log_file: str = "../logs/feature_extraction.log"):
        """Initializes the data folder
        """
        self.preprocessor_state_obj = preprocessor_state_obj
        self.logger = make_logger("FeatureExtraction",log_file)

    def extract_column_types(self,dataset:pd.DataFrame,col_type:str):
        col_list = dataset.select_dtypes(include=[col_type]).columns
        self.logger.info(f"Columns of type {col_type}: {col_list.tolist()}")
        return col_list
    
    def sanitize_names(self, raw_names):
        """Converts sklearn names to clean UPPER_SNAKE_CASE."""
        return [str(name).replace("remainder__", "").replace("-", "_").replace(" ", "_").upper() for name in raw_names]
    
    
    def fit_transform(self,dataset:pd.DataFrame)->pd.DataFrame:
        """Function to fit the feature engineering preprocessor on train set"""
        self.logger.info(f"Fit & Transform Started")
        self.logger.info(f"Loaded Dataset Shape {dataset.shape}")

        try:
            X = dataset.copy()
            # Extract Numerical Features
            num_cols = self.extract_column_types(dataset, col_type="number")
            # Extract categorical columns
            cat_cols = self.extract_column_types(dataset, col_type="object")
            
            self.logger.info(f"🔢 Numerical cols: {num_cols}")
            self.logger.info(f"🏷️  Categorical cols: {cat_cols}")
            
            # Build Transformer
            transformers = []

            if len(num_cols)>0:
                num_pipeline = Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                    ("scaler", StandardScaler())
                ])
                transformers.append(("num", num_pipeline, num_cols))
                
                
            if len(cat_cols)>0:
                transformers.append(("cat", OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols))

            # Initialize Column Transformer
            preprocessor = ColumnTransformer(transformers=transformers, remainder="drop",sparse_threshold=0)
            

            # Fit Preprocessor
            X_transformed = preprocessor.fit_transform(X)
            self.logger.info(f"Transformed dataframe : {X_transformed.shape}")
            # Reconstruct DataFrame
            feature_names = self.sanitize_names(preprocessor.get_feature_names_out())
            self.logger.info(f"Feature Names {feature_names}")
            
            X_processed = pd.DataFrame(data=X_transformed, columns=feature_names, index=X.index)
            self.logger.info(f"Shape of processed dataframe : {X_processed.shape}")

            preprocessor_state_obj = PreprocessorState(
                transformer=preprocessor, 
                target_col= None,
                logger = self.logger
            )
            self.preprocessor_state_obj = preprocessor_state_obj
        except Exception as e:
            self.logger.exception(f"Feature Engineering Error during fit transform, {str(e)}")
            raise

        return X_processed, preprocessor_state_obj
        

    def transform(self,dataset:pd.DataFrame, target_col: str = None)->pd.DataFrame:
        """Function to fit the feature engineering preprocessor on train set"""
        self.logger.info(f"Transform Started")
        self.logger.info(f"Loaded Dataset Shape: {dataset.shape}")

        try:
            X = dataset.copy()
            # Extract Numerical Features
            num_cols = self.extract_column_types(dataset, col_type="number")
            # Extract categorical columns
            cat_cols = self.extract_column_types(dataset, col_type="object")
            
            self.logger.info(f"🔢 Numerical cols: {num_cols}")
            self.logger.info(f"🏷️  Categorical cols: {cat_cols}")


            # Apply Transformation
            if self.preprocessor_state_obj.transformer is None:
                raise ValueError("Inference Mode: No pre-fitted transformer provided.")
            
            X_transformed = self.preprocessor_state_obj.transformer.transform(X)

            feature_names = self.sanitize_names(self.preprocessor_state_obj.transformer.get_feature_names_out())
            self.logger.info(f"Feature Names {feature_names}")

            X_processed = pd.DataFrame(data=X_transformed, columns=feature_names, index=X.index)
            self.logger.info(f"Shape of processed dataframe {X_processed.shape}")
            return X_processed
        
        except Exception as e:
            self.logger.exception(f"Feature Engineering Error during transform - {str(e)}")
            raise

In [101]:
# Load Training Dataset
load_dataset = LoadDataset(data_folder = data_folder_path)
train_dataset = load_dataset.load_excel(file_name = "train_dataset.xlsx")
print(train_dataset.shape)

X_train,y_train = load_dataset.clean_data(train_dataset,target_col = "Churn")
print(X_train.shape,y_train.shape)

(4928, 21)
(4928, 19) (4928,)


In [102]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4928 entries, 0 to 4927
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   GENDER            4928 non-null   object 
 1   SENIORCITIZEN     4928 non-null   int64  
 2   PARTNER           4928 non-null   object 
 3   DEPENDENTS        4928 non-null   object 
 4   TENURE            4928 non-null   int64  
 5   PHONESERVICE      4928 non-null   object 
 6   MULTIPLELINES     4928 non-null   object 
 7   INTERNETSERVICE   4928 non-null   object 
 8   ONLINESECURITY    4928 non-null   object 
 9   ONLINEBACKUP      4928 non-null   object 
 10  DEVICEPROTECTION  4928 non-null   object 
 11  TECHSUPPORT       4928 non-null   object 
 12  STREAMINGTV       4928 non-null   object 
 13  STREAMINGMOVIES   4928 non-null   object 
 14  CONTRACT          4928 non-null   object 
 15  PAPERLESSBILLING  4928 non-null   object 
 16  PAYMENTMETHOD     4928 non-null   object 


In [103]:
# Feature Engineering Dataset
feature_extractor = FeatureExtraction()
X_train_transformed, preprocessor = feature_extractor.fit_transform(dataset = X_train)

In [104]:
X_train_transformed

,NUM__SENIORCITIZEN,NUM__TENURE,NUM__MONTHLYCHARGES,NUM__TOTALCHARGES,CAT__GENDER_MALE,CAT__PARTNER_YES,CAT__DEPENDENTS_YES,CAT__PHONESERVICE_YES,CAT__MULTIPLELINES_NO_PHONE_SERVICE,CAT__MULTIPLELINES_YES,...,CAT__STREAMINGTV_NO_INTERNET_SERVICE,CAT__STREAMINGTV_YES,CAT__STREAMINGMOVIES_NO_INTERNET_SERVICE,CAT__STREAMINGMOVIES_YES,CAT__CONTRACT_ONE_YEAR,CAT__CONTRACT_TWO_YEAR,CAT__PAPERLESSBILLING_YES,CAT__PAYMENTMETHOD_CREDIT_CARD,CAT__PAYMENTMETHOD_ELECTRONIC_CHECK,CAT__PAYMENTMETHOD_MAILED_CHECK
0,-0.436278,-1.149113,-1.460183,-0.959042,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.436278,1.256801,-0.821342,0.065004,1.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,2.292118,-0.211214,0.798931,0.097209,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,2.292118,1.623805,1.507283,2.502871,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4,-0.436278,-1.189891,-1.137453,-0.954125,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4923,-0.436278,1.297580,-0.136160,0.707495,1.0,1.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
4924,-0.436278,-1.271448,-1.460183,-0.987060,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4925,2.292118,1.460692,-0.369519,0.623684,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0
4926,-0.436278,1.093689,1.553624,1.907845,1.0,1.0,1.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0


In [105]:
X_train_transformed.describe()

,NUM__SENIORCITIZEN,NUM__TENURE,NUM__MONTHLYCHARGES,NUM__TOTALCHARGES,CAT__GENDER_MALE,CAT__PARTNER_YES,CAT__DEPENDENTS_YES,CAT__PHONESERVICE_YES,CAT__MULTIPLELINES_NO_PHONE_SERVICE,CAT__MULTIPLELINES_YES,...,CAT__STREAMINGTV_NO_INTERNET_SERVICE,CAT__STREAMINGTV_YES,CAT__STREAMINGMOVIES_NO_INTERNET_SERVICE,CAT__STREAMINGMOVIES_YES,CAT__CONTRACT_ONE_YEAR,CAT__CONTRACT_TWO_YEAR,CAT__PAPERLESSBILLING_YES,CAT__PAYMENTMETHOD_CREDIT_CARD,CAT__PAYMENTMETHOD_ELECTRONIC_CHECK,CAT__PAYMENTMETHOD_MAILED_CHECK
count,4.928000e+03,4.928000e+03,4.928000e+03,4.928000e+03,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,...,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000,4928.000000
mean,-1.658125e-17,-8.506904e-17,-8.506904e-17,-1.182315e-16,0.509334,0.484375,0.300325,0.900365,0.099635,0.418222,...,0.224026,0.379870,0.224026,0.382508,0.210836,0.237825,0.593141,0.209213,0.335024,0.235998
std,1.000101e+00,1.000101e+00,1.000101e+00,1.000101e+00,0.499964,0.499807,0.458446,0.299543,0.299543,0.493317,...,0.416982,0.485404,0.416982,0.486049,0.407944,0.425794,0.491298,0.406788,0.472047,0.424664
min,-4.362776e-01,-1.312226e+00,-1.519764e+00,-9.959418e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-4.362776e-01,-9.452221e-01,-1.011670e+00,-8.310168e-01,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-4.362776e-01,-1.296578e-01,1.948454e-01,-3.878485e-01,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
75%,-4.362776e-01,9.305757e-01,8.436166e-01,6.462148e-01,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,...,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000
max,2.292118e+00,1.623805e+00,1.806843e+00,2.851204e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [106]:
X_train_transformed.to_csv("../data/X_train_transformed.csv",index=False)
pd.DataFrame(y_train).to_csv("../data/y_train.csv",index=False)

In [111]:
# Load Testing Dataset
load_dataset = LoadDataset(data_folder = data_folder_path)
test_dataset = load_dataset.load_excel(file_name = "test_dataset.xlsx")
print(test_dataset.shape)

X_test,y_test = load_dataset.clean_data(test_dataset,target_col = "Churn")
print(X_test.shape,y_test.shape)

(1410, 21)
(1410, 19) (1410,)


In [112]:
preprocessor.save(file_path = "./model/preprocessor_v1.pkl")

True

In [113]:
X_test_transformed = feature_extractor.transform(dataset = X_test)

In [114]:
X_test_transformed.describe()

,NUM__SENIORCITIZEN,NUM__TENURE,NUM__MONTHLYCHARGES,NUM__TOTALCHARGES,CAT__GENDER_MALE,CAT__PARTNER_YES,CAT__DEPENDENTS_YES,CAT__PHONESERVICE_YES,CAT__MULTIPLELINES_NO_PHONE_SERVICE,CAT__MULTIPLELINES_YES,...,CAT__STREAMINGTV_NO_INTERNET_SERVICE,CAT__STREAMINGTV_YES,CAT__STREAMINGMOVIES_NO_INTERNET_SERVICE,CAT__STREAMINGMOVIES_YES,CAT__CONTRACT_ONE_YEAR,CAT__CONTRACT_TWO_YEAR,CAT__PAPERLESSBILLING_YES,CAT__PAYMENTMETHOD_CREDIT_CARD,CAT__PAYMENTMETHOD_ELECTRONIC_CHECK,CAT__PAYMENTMETHOD_MAILED_CHECK
count,1410.000000,1410.000000,1410.000000,1410.000000,1410.000000,1410.000000,1410.000000,1410.000000,1410.000000,1410.000000,...,1410.000000,1410.000000,1410.000000,1410.000000,1410.000000,1410.000000,1410.000000,1410.00000,1410.000000,1410.000000
mean,0.032000,0.005720,0.035742,0.028105,0.489362,0.470922,0.295745,0.904255,0.095745,0.425532,...,0.202128,0.382979,0.202128,0.383688,0.204255,0.247518,0.577305,0.22766,0.343262,0.216312
std,1.029133,1.009160,0.982541,1.010239,0.500064,0.499331,0.456539,0.294345,0.294345,0.494599,...,0.401729,0.486286,0.401729,0.486456,0.403299,0.431723,0.494163,0.41947,0.474967,0.411876
min,-0.436278,-1.312226,-1.501559,-0.995942,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
25%,-0.436278,-0.945222,-0.801068,-0.817008,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
50%,-0.436278,-0.170436,0.222981,-0.377372,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.00000,0.000000,0.000000
75%,-0.436278,0.930576,0.846927,0.732794,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,...,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.00000,1.000000,0.000000
max,2.292118,1.623805,1.765467,2.844692,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000


In [115]:
X_test_transformed.to_csv("../data/X_test_transformed.csv",index=False)
pd.DataFrame(y_test).to_csv("../data/y_test.csv",index=False)